# Смена директории


In [1]:
%cd ..

/Users/uzumnasiya/HSE/Year_Project


# Импорт библиотек


In [2]:
import logging
import warnings
import time


import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import yaml
import joblib
import shap
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from category_encoders import CatBoostEncoder
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


from utils.dev_utils import get_pool
from utils.pipeline_utils import CustomPreprocessor
from utils.metrics import MetricCalculator, metric_funcs
from utils.style.styler import style_metrics
from utils.style.html_output import print_multiple_html
from utils.plot_utils import plot_gini_by_period_styled, plot_roc_by_masks
import os
from pathlib import Path

import mlflow
import mlflow.catboost
from dotenv import load_dotenv
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient


In [3]:
logging.getLogger().setLevel(logging.WARNING)
warnings.filterwarnings('ignore')
sns.set_palette('bright')


pd.options.display.float_format = "{:.2f}".format
pd.options.display.max_rows = 100
pd.options.display.max_columns = 100

In [4]:
%load_ext autoreload
%autoreload 2
%aimport utils.plot_utils
%aimport utils.eda_utils
%aimport utils.style_utils
%aimport utils.psi
%aimport utils.style.styler

# Входные данные


## Загрузка данных


In [5]:
data = pd.read_parquet('./data/processed/data.pqt')

data.shape

(1097231, 633)

## Валидные переменные


In [6]:
path = r'./docs/valid_features.xlsx'
valid_features_data = pd.read_excel(path, index_col=False)

## Конфиги/константы


In [7]:
TARGET = 'target'
DATE_MONTH = 'date_month'
DATE_QUARTAL = 'date_quartal'

FEATURES = valid_features_data.loc[(
    valid_features_data['valid_flag'] == 1)]['attribute'].values
CAT_FEATURES = sorted(set(FEATURES) & set(
    data.select_dtypes(include=["object", "category"]).columns))
NUM_FEATURES = sorted(set(FEATURES) & set(
    data.select_dtypes(include=["number"]).columns))

TRAIN_MASK = (data['sample_type'] == 'TRAIN')
TEST_MASK = (data['sample_type'] == 'TEST')
OOT_MASK = (data['sample_type'] == 'OOT')

DEV_MASK = (data['competition_sample_type'] == 'TRAIN')

In [8]:
print_multiple_html(
    ('Кол-во переменных: ', len(FEATURES)),
    ('Кол-во категориальных переменных: ', len(CAT_FEATURES))
)

In [9]:
metr_funcs = {
    'roc_auc': metric_funcs.roc_auc_score_nan,
    'gini': metric_funcs.gini_score_nan,
    'precision': metric_funcs.precision,
    'recall': metric_funcs.recall
}

stats_funcs = {
    'obs_cnt': lambda y_true, data: len(y_true),
    'target_cnt': lambda y_true, data: sum(y_true),
    'DR': lambda y_true, data: sum(y_true) / len(y_true)
}

# Класс для расчета метрик
metr_calc = MetricCalculator(metr_funcs=metr_funcs, stats_funcs=stats_funcs)


STYLE_CONFIG = {
    'percent_cols': ['gini', 'roc_auc', 'DR', 'precision', 'recall'],
    'int_cols': ['obs_cnt', 'target_cnt'],
    'gradient_cols': ['gini', 'roc_auc', 'precision', 'recall'],
    'gradient_cmap': 'RdYlGn',
}

# Словарь для сохранения времени обучения моделей
train_timing = {}

In [10]:
def metrics_split(data, group_cols, metr_calc, pred_cols=['lg_model_preds'], target='target', asc=True):
    """Упрощенный интерфес для расчета метрик"""

    metr_split = metr_calc.calculate(
        data, true_col=target, pred_cols=pred_cols, group_cols=group_cols)
    metr_split = (
        metr_split
        .sort_values(by=group_cols, ascending=asc)
        .set_index(group_cols)
    )

    return metr_split

# MLflow и финальная CatBoost


## Подключение к трекингу


In [11]:
load_dotenv()
os.environ.setdefault("AWS_ACCESS_KEY_ID", os.getenv("AWS_ACCESS_KEY_ID", "admin"))
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", os.getenv("AWS_SECRET_ACCESS_KEY", "password"))
raw_s3 = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")
if "minio:9000" in raw_s3:
    raw_s3 = "http://localhost:9000"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = raw_s3

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5050")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.search_experiments(max_results=5)


[<Experiment: artifact_location='mlflow-artifacts:/', creation_time=1780520067164, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1780520067164, lifecycle_stage='active', name='year-project-fraud-catboost', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='s3://mlflow-bucket/mlflow/0', creation_time=1780519760478, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1780519760478, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

## Финальные признаки и параметры


In [12]:
with open(r'./models/params/features.yaml', encoding='utf-8') as file:
    features_cfg = yaml.safe_load(file)

with open(r'./models/params/best_params.yaml', encoding='utf-8') as file:
    final_params = yaml.safe_load(file)

FINAL_FEATURES = features_cfg['FINAL_FEATURES']
FINAL_CAT_FEATURES = features_cfg['FINAL_CAT_FEATURES']
ARTIFACT_DIR = Path('./reports/mlflow_artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print_multiple_html(
    ('Кол-во финальных фичей: ', len(FINAL_FEATURES)),
    ('Кол-во финальных категориальных фичей: ', len(FINAL_CAT_FEATURES)),
)


## Выбор модели

В 3.2 лучшие метрики на TEST/OOT у CatBoost после Optuna по топ-30 признакам. Baseline для сравнения — CatBoost с дефолтными параметрами на тех же фичах.


## Подготовка выборки


In [13]:
dev_sample = data.loc[DEV_MASK].copy()
dev_sample = CustomPreprocessor(cat_features=FINAL_CAT_FEATURES).transform(dev_sample)

train_pool = get_pool(dev_sample, TARGET, TRAIN_MASK, FINAL_FEATURES, FINAL_CAT_FEATURES)
test_pool = get_pool(dev_sample, TARGET, TEST_MASK, FINAL_FEATURES, FINAL_CAT_FEATURES)
oot_pool = get_pool(dev_sample, TARGET, OOT_MASK, FINAL_FEATURES, FINAL_CAT_FEATURES)


## Baseline CatBoost


In [14]:
default_params = {
    'iterations': 100,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'early_stopping_rounds': 20,
    'verbose': 0,
    'random_seed': 42,
}

baseline_model = CatBoostClassifier(**default_params, cat_features=FINAL_CAT_FEATURES)
baseline_model.fit(train_pool, eval_set=test_pool, use_best_model=True, plot=False)
dev_sample['baseline_preds'] = baseline_model.predict_proba(dev_sample[FINAL_FEATURES])[:, 1]


## Финальная модель


In [15]:
final_model = CatBoostClassifier(**final_params, cat_features=FINAL_CAT_FEATURES)
final_model.fit(train_pool, eval_set=test_pool, use_best_model=True, plot=False)
dev_sample['final_model_preds'] = final_model.predict_proba(dev_sample[FINAL_FEATURES])[:, 1]


## Метрики train / test / oot


In [22]:
metr_split = metrics_split(
    data=dev_sample,
    target=TARGET,
    pred_cols=['baseline_preds', 'final_model_preds'],
    group_cols='sample_type',
    metr_calc=metr_calc,
    asc=False,
)


(metr_split
    .pivot_table(index='level_1', 
                 columns=metr_split.index, 
                 values='roc_auc')
    .style
    .format("{:.1%}")
    .background_gradient(cmap='RdYlGn', axis=None)
)

sample_type,OOT,TEST,TRAIN
level_1,,,
baseline_preds,91.1%,94.3%,96.1%
final_model_preds,92.3%,95.9%,98.2%


## Логирование в MLflow (тег PRD)


In [23]:
EXPERIMENT_NAME = 'year-project-fraud-catboost'
REGISTERED_MODEL_NAME = 'year_project_fraud_catboost'
client = MlflowClient()

exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if exp is None:
    exp_id = client.create_experiment(name=EXPERIMENT_NAME, artifact_location='mlflow-artifacts:/')
else:
    exp_id = exp.experiment_id

mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(experiment_id=exp_id, run_name='catboost_final_prd') as run:
    mlflow.set_tag('env', 'PRD')

    mlflow.log_params(final_params)
    mlflow.log_param('n_features', len(FINAL_FEATURES))

    for split_name, mask in [('train', TRAIN_MASK), ('test', TEST_MASK), ('oot', OOT_MASK)]:
        sub = dev_sample.loc[mask]
        for prefix, col in [('final', 'final_model_preds'), ('baseline', 'baseline_preds')]:
            proba = sub[col].values
            y = sub[TARGET].values
            mlflow.log_metric(f'{prefix}_roc_auc_{split_name}', roc_auc_score(y, proba))
            mlflow.log_metric(f'{prefix}_gini_{split_name}', 2 * roc_auc_score(y, proba) - 1)

    input_df = dev_sample.loc[TEST_MASK, FINAL_FEATURES].head(5).copy()
    for col in FINAL_CAT_FEATURES:
        input_df[col] = input_df[col].astype(str)

    signature = infer_signature(
        input_df,
        dev_sample.loc[TEST_MASK, 'final_model_preds'].values[:5],
    )

    model_info = mlflow.catboost.log_model(
        cb_model=final_model,
        artifact_path='model',
        signature=signature,
        input_example=input_df,
        registered_model_name=REGISTERED_MODEL_NAME,
    )

    preds_sample = dev_sample.loc[TEST_MASK, FINAL_FEATURES + [TARGET, 'final_model_preds']].head(200)
    preds_path = ARTIFACT_DIR / 'predictions_sample.csv'
    preds_sample.to_csv(preds_path, index=False)
    mlflow.log_artifact(str(preds_path), artifact_path='samples')

    full_preds = CustomPreprocessor(cat_features=FINAL_CAT_FEATURES).transform(data.copy())
    full_preds['final_model_preds'] = final_model.predict_proba(full_preds[FINAL_FEATURES])[:, 1]

    fig, axes = plt.subplots(1, 2, figsize=(16, 4), gridspec_kw={'width_ratios': [1, 1.7]})
    masks_dict = {'TRAIN': TRAIN_MASK, 'TEST': TEST_MASK, 'OOT': OOT_MASK}
    plot_roc_by_masks(full_preds, TARGET, 'final_model_preds', masks_dict,
                      title='ROC-кривые по различным подвыборкам', ax=axes[0])
    metr_month = metrics_split(
        data=full_preds.loc[DEV_MASK], target=TARGET, pred_cols='final_model_preds',
        group_cols=DATE_MONTH, metr_calc=metr_calc, asc=True,
    )
    plot_gini_by_period_styled(metr_month.reset_index(), 'gini', DATE_MONTH,
                               target_gini=40, title='Динамика Gini по периодам', ax=axes[1])
    plt.tight_layout()
    plot_path = ARTIFACT_DIR / 'roc_gini.png'
    plt.savefig(plot_path, dpi=120, bbox_inches='tight')
    plt.close()
    mlflow.log_artifact(str(plot_path), artifact_path='plots')

    version = model_info.registered_model_version
    client.set_model_version_tag(REGISTERED_MODEL_NAME, version, 'env', 'PRD')
    client.set_registered_model_alias(REGISTERED_MODEL_NAME, 'prd', version)
    run_id = run.info.run_id

print('run_id:', run_id)


2026/06/04 00:22:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'year_project_fraud_catboost' already exists. Creating a new version of this model...
2026/06/04 00:22:22 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: year_project_fraud_catboost, version 2
Created version '2' of model 'year_project_fraud_catboost'.


🏃 View run catboost_final_prd at: http://localhost:5050/#/experiments/1/runs/b15f6693db8d494ba4d339a936c32ddf
🧪 View experiment at: http://localhost:5050/#/experiments/1
run_id: b15f6693db8d494ba4d339a936c32ddf


## Анализ ошибок


In [24]:
THRESHOLD = 0.5
test_df = dev_sample.loc[TEST_MASK].copy()
test_df['pred'] = (test_df['final_model_preds'] >= THRESHOLD).astype(int)
test_df['error_type'] = 'OK'
test_df.loc[(test_df[TARGET] == 0) & (test_df['pred'] == 1), 'error_type'] = 'FP'
test_df.loc[(test_df[TARGET] == 1) & (test_df['pred'] == 0), 'error_type'] = 'FN'

errors = test_df[test_df['error_type'].isin(['FP', 'FN'])].copy()
errors['amt_bucket'] = pd.qcut(
    errors['TransactionAmt'].fillna(errors['TransactionAmt'].median()),
    q=3, labels=['low', 'mid', 'high'], duplicates='drop',
)

cols_show = ['TransactionAmt', 'P_emaildomain_new', 'card_addr1_pair', 'V294',
             TARGET, 'final_model_preds', 'error_type']
error_examples = (
    errors.sort_values('final_model_preds', ascending=False)
    .groupby('error_type', group_keys=False).head(10)[cols_show].head(20)
)
error_examples

error_path = ARTIFACT_DIR / 'error_examples.csv'
error_examples.to_csv(error_path, index=False)
with mlflow.start_run(run_id=run_id):
    mlflow.log_artifact(str(error_path), artifact_path='error_analysis')


🏃 View run catboost_final_prd at: http://localhost:5050/#/experiments/1/runs/b15f6693db8d494ba4d339a936c32ddf
🧪 View experiment at: http://localhost:5050/#/experiments/1


## Устойчивость (robustness)


In [25]:
rng = np.random.default_rng(42)
sample = dev_sample.loc[TEST_MASK, FINAL_FEATURES].head(500).copy()
base_proba = final_model.predict_proba(sample)[:, 1]

perturbed = sample.copy()
num_cols = [c for c in FINAL_FEATURES if c not in FINAL_CAT_FEATURES]
for col in num_cols:
    noise = rng.normal(0, 0.01 * perturbed[col].astype(float).std(), size=len(perturbed))
    perturbed[col] = perturbed[col].astype(float) + noise

delta = np.abs(final_model.predict_proba(perturbed)[:, 1] - base_proba)
pd.DataFrame({
    'mean_abs_delta': [delta.mean()],
    'p95_abs_delta': [np.quantile(delta, 0.95)],
    'share_delta_gt_0.05': [(delta > 0.05).mean()],
})


,mean_abs_delta,p95_abs_delta,share_delta_gt_0.05
0,0.03,0.14,0.13
